In [1]:
import glob
import pandas as pd

In [2]:
combined_df = pd.DataFrame()

# Use correct path relative to notebook location (in src/)
for file in glob.glob("../results/results_*.csv"):
    df = pd.read_csv(file)
    combined_df = pd.concat([combined_df, df], ignore_index=True)

print(f"Loaded {len(combined_df)} rows from results files")

Loaded 238 rows from results files


In [3]:
combined_df.head()

,model,species,training_size,n_folds,n_epochs,batch_size,test_size_per_class,cv_auc_mean,cv_auc_std,cv_auc_ci_95,cv_auc_n,test_auc_mean,test_auc_std,test_auc_ci_95,test_auc_n
0,birdnet,bullfrog,0,0,0,32,50,NaN,NaN,NaN,0,0.3320,NaN,NaN,1
1,birdnet,bullfrog,10,5,20,32,50,1.0000,0.0000,0.0000,5,0.9982,0.0016,0.0014,5
2,birdnet,bullfrog,20,5,20,32,50,0.9925,0.0168,0.0147,5,0.9988,0.0012,0.0011,5
3,birdnet,bullfrog,40,5,20,32,50,0.9850,0.0075,0.0066,5,0.9993,0.0004,0.0004,5
4,birdnet,bullfrog,60,5,20,32,50,0.9956,0.0051,0.0045,5,0.9998,0.0002,0.0002,5


In [4]:
# Check what columns are in the dataframe
print("Columns in combined_df:")
print(combined_df.columns.tolist())
print("\nFirst few rows:")
print(combined_df.head())

Columns in combined_df:
['model', 'species', 'training_size', 'n_folds', 'n_epochs', 'batch_size', 'test_size_per_class', 'cv_auc_mean', 'cv_auc_std', 'cv_auc_ci_95', 'cv_auc_n', 'test_auc_mean', 'test_auc_std', 'test_auc_ci_95', 'test_auc_n']

First few rows:
     model   species  training_size  n_folds  n_epochs  batch_size  \
0  birdnet  bullfrog              0        0         0          32   
1  birdnet  bullfrog             10        5        20          32   
2  birdnet  bullfrog             20        5        20          32   
3  birdnet  bullfrog             40        5        20          32   
4  birdnet  bullfrog             60        5        20          32   

   test_size_per_class  cv_auc_mean  cv_auc_std  cv_auc_ci_95  cv_auc_n  \
0                   50          NaN         NaN           NaN         0   
1                   50       1.0000      0.0000        0.0000         5   
2                   50       0.9925      0.0168        0.0147         5   
3                 

In [5]:
# Check what models we have
print("Unique models:", sorted(combined_df['model'].unique()))

Unique models: ['birdnet', 'mobilenet', 'perch', 'resnet', 'vgg']


In [39]:
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for dev container
import matplotlib.pyplot as plt
import seaborn as sns

# Set publication-ready style
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif'],  # Available on Ubuntu by default
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 100,  # Lower for notebook display
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.5,
    'lines.linewidth': 2,
    'lines.markersize': 6,
})

def plot_species_models_publication(df, output_path="results/species_comparison.png"):
    """Create publication-quality plots with 95% confidence intervals."""
    
    # Color palette for models - updated for actual model names
    colors = {
        'birdnet': '#2E86AB',    # Professional blue
        'mobilenet': '#A23B72',  # Deep magenta
        'perch': '#F18F01',      # Warm orange
        'resnet': '#06A77D',     # Teal green
        'vgg': '#C73E1D',        # Deep red
    }
    
    markers = {
        'birdnet': 'o',
        'mobilenet': 's',
        'perch': '^',
        'resnet': 'D',
        'vgg': 'v',
    }
    
    species_list = sorted(df["species"].unique())
    n_species = len(species_list)
    
    # Create 2 rows x 3 columns grid with extra space at bottom for legend
    fig, axes = plt.subplots(4, 2, figsize=(12, 10.5))
    axes = axes.flatten()  # Flatten to make iteration easier
    
    # Store handles and labels for shared legend
    handles, labels = None, None
    
    for i, species in enumerate(['bullfrog', 'coyote', 'pacific_chorus_frog', 'human_vocal', 'woodhouses_toad', 'engine', 'field_cricket']):
        ax = axes[i]
        species_data = df[df["species"] == species]
        models = sorted(species_data["model"].unique())
        
        for model in models:
            model_data = species_data[species_data["model"] == model].sort_values("training_size")
            
            line = ax.errorbar(
                model_data["training_size"],
                model_data["test_auc_mean"],
                yerr=model_data["test_auc_ci_95"],
                label=model.upper(),
                marker=markers.get(model, 'o'),
                color=colors.get(model, '#333333'),
                capsize=4,
                capthick=1.5,
                elinewidth=1.5,
                alpha=0.9,
                markeredgewidth=0.5,
                markeredgecolor='white',
            )
        
        # Get handles and labels from first subplot for shared legend
        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
        
        species_name = species.replace('_', ' ').title()
        ax.set_title(species_name, fontweight='semibold', pad=10)
        ax.set_xlabel("Training Size (samples per class)", fontweight='medium')
        ax.set_ylabel("Test ROC-AUC", fontweight='medium')
        
        ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.5)
        ax.set_axisbelow(True)
        ax.set_ylim(0.2, 1.02)
        ax.set_xlim(-5, 120)
        ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5, zorder=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    # Use the 8th subplot for the legend
    ax_legend = axes[7]
    ax_legend.axis('off')  # Hide axis
    ax_legend.legend(handles, labels, 
                     loc='center',
                     ncol=1,  # Stack vertically
                     frameon=True, 
                     fancybox=False, 
                     edgecolor='gray', 
                     framealpha=0.95,
                     fontsize=11)
    
    # Adjust subplot spacing
    plt.subplots_adjust(hspace=0.8, wspace=0.25)
    
    plt.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0.3, facecolor='white')
    print(f"✓ Plot saved to {output_path}")
    plt.close()  # Close instead of show in headless environment
    return fig

In [40]:
plot_species_models_publication(combined_df, "../results/species_comparison.png")

✓ Plot saved to ../results/species_comparison.png


<Figure size 1200x1050 with 8 Axes>